# 04 — Chronological Validation

This notebook defines a reproducible training, validation, and test split for the stock-ranking experiment. Each theory section is followed by an empty code cell for your implementation. The objective is to check the information timeline before fitting models.

## Theory before implementation

Imagine training with outcomes from 2024 and evaluating predictions on 2020 stocks. A random row split can create that scenario, even though future observations would not have been available when making the earlier decisions. Chronological validation measures performance using a more realistic information timeline.

Training learns model parameters. Validation compares planned choices, such as regularization settings. Test evaluates the selected approach after those choices are frozen. Repeatedly adjusting a model using test results turns that period into another validation set.

Start with training in 2018–2021, validation in 2022, and test in 2023–2024. Assign all stocks on one prediction date together. EDA already examined the full sample, so describe test data as held out from model fitting and tuning rather than entirely unseen.

A December training row may have a five-day outcome ending in January, inside validation. Keep training rows only when label_end_date is strictly before validation begins; apply the equivalent rule between validation and test. Removing those overlapping labels is called purging. Actual label dates account for weekends and holidays better than a calendar-day gap.

Rolling features may use earlier history across split boundaries because that history was available. A scaler or imputer learning statistics across dates must instead fit on training only, then apply unchanged to later periods.

Begin with a transparent momentum-ranking baseline. Evaluate scores across stocks on each date using Spearman correlation, which measures agreement in ordering. Overlapping outcomes complicate uncertainty estimates; predictive ranking also does not establish profitability after trading costs.

The output is a reproducible split manifest, boundary exclusions, verified chronology, and a written evaluation plan.

## Implementation

The following sections build and check the splits before model fitting.

## 1. Why chronological validation is needed

Imagine training with outcomes from 2024 and then asking the model to predict stocks in 2020. A random row split can create exactly that situation. It measures performance under information the model would not have had at the prediction time.

This project compares stocks on the same date, so every stock from a prediction date must stay in the same split. This notebook prepares and verifies those splits before any model is fitted.

In [ ]:
import numpy as np
import pandas as pd

SyntaxError: invalid syntax (1381439962.py, line 3)

## 2. Load the labeled data and inspect its keys

Load LABELED_DATA_PATH, parsing both date and label_end_date as dates. Sort by date and symbol. Verify unique (date, symbol) keys, available targets, valid label end dates, and daily universe coverage.

Keep the target and input lists explicit. The first experiment can use the four raw features: five_day_return, twenty_day_return, volatility_20, and relative_volume_20. Its target is target_percentile_5d. Forward returns, target percentiles, label_end_date, and any future-price columns must not enter the input list.

In [15]:
from factor_model.config import LABELED_DATA_PATH

df = pd.read_csv(LABELED_DATA_PATH, parse_dates=["date", "label_end_date"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17360 entries, 0 to 17359
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      17360 non-null  datetime64[us]
 1   symbol                    17360 non-null  str           
 2   open                      17360 non-null  float64       
 3   high                      17360 non-null  float64       
 4   low                       17360 non-null  float64       
 5   close                     17360 non-null  float64       
 6   adjusted_close            17360 non-null  float64       
 7   volume                    17360 non-null  int64         
 8   normalized_price          17360 non-null  float64       
 9   dollar_volume             17360 non-null  float64       
 10  daily_return              17360 non-null  float64       
 11  five_day_return           17360 non-null  float64       
 12  twenty_day_return         173

## 3. Define training, validation, and test periods

Use an initial fixed plan: training prediction dates before January 1, 2022; validation prediction dates from January 1, 2022 up to January 1, 2023; and test prediction dates from January 1, 2023 onward. Only available labeled observations belong to these periods; the current export ends before the final raw price date.

Training estimates model parameters. Validation compares a small set of choices, such as regularization strength. Test evaluates the selected approach after those choices are frozen. Calendar boundary dates need not be trading days; date comparisons still assign observed trading dates correctly.

Declare boundary dates and experiment metadata in this cell. Do not change the boundaries in response to test results. The earlier EDA already inspected the full historical period, so describe the test as held out from model fitting and tuning, rather than claiming it has never been viewed.

In [19]:
validation_start = pd.Timestamp("2022-01-01")
test_start = pd.Timestamp("2023-01-01")


## 4. Build preliminary splits by prediction date

Create separate training, validation, and test tables using the declared boundaries. Assign all rows for a date together, preserving their (date, symbol) identities.

These preliminary splits use the date of the features. They still need a label-timing check: an outcome attached to a December training row may not become known until January in the next period. Keep preliminary copies so you can inspect which rows are subsequently removed.

In [20]:
train_df = df[df["date"] < validation_start].copy()

valid_df = df[
    (df["date"] >= validation_start) &
    (df["date"] < test_start)
].copy()

test_df = df[df["date"] >= test_start].copy()


## 5. Remove labels crossing the next period

Suppose a prediction is dated December 30 but its five-day outcome ends in January, after validation starts. That outcome would not have been available before the validation period. Remove that row from training.

For this initial experiment, retain training rows only when label_end_date is strictly before the validation boundary. Retain validation rows only when label_end_date is strictly before the test boundary. This removes overlapping labels at the boundaries and is often called purging. Use actual label end dates, rather than subtracting five calendar days, because weekends and holidays affect the horizon.

The test set contains only already observable labels from notebook 03. Keep a list of removed keys and the reason for removal. This experiment does not add a separate post-period embargo; more elaborate walk-forward schemes can be considered later.

In [22]:
train_df = train_df[train_df["label_end_date"] <= validation_start].copy()
valid_df = valid_df[valid_df["label_end_date"] <= test_start].copy()

## 6. Verify chronology, disjoint rows, and coverage

Assert that each split is non-empty, its keys are unique, and no keys or prediction dates are shared between splits. Confirm that the latest training label end date precedes the validation boundary and the latest validation label end date precedes the test boundary.

Summarize each split's row count, date count, first and last prediction date, latest label end date, and symbols per date. Confirm that the intended ten-symbol universe remains present on every retained date. Inspect the removed boundary observations to make the five-day overlap visible.

Check required input and target values are finite. Do not silently drop individual stocks here to repair missing data, because that would change the daily comparison universe.

In [23]:
feature_columns = [
    "five_day_return",
    "twenty_day_return",
    "volatility_20",
    "relative_volume_20",
]
target_column = "target_percentile_5d"

splits = {"train": train_df, "validation": valid_df, "test": test_df}
expected_symbols = set(df["symbol"])
summary = []

for name, split in splits.items():
    assert not split.empty, f"{name}: split is empty"
    assert not split.duplicated(["date", "symbol"]).any(), f"{name}: duplicate keys"
    assert split[["date", "symbol", "label_end_date"]].notna().all().all(), (
        f"{name}: missing keys or label dates"
    )
    assert np.isfinite(split[feature_columns + [target_column]].to_numpy()).all(), (
        f"{name}: missing or infinite model inputs/targets"
    )
    assert (split["label_end_date"] > split["date"]).all(), (
        f"{name}: outcome must end after prediction date"
    )

    daily_symbols = split.groupby("date")["symbol"].agg(set)
    assert daily_symbols.apply(lambda symbols: symbols == expected_symbols).all(), (
        f"{name}: incomplete daily universe"
    )

    summary.append({
        "split": name,
        "rows": len(split),
        "dates": split["date"].nunique(),
        "first_date": split["date"].min(),
        "last_date": split["date"].max(),
        "latest_label_end": split["label_end_date"].max(),
        "symbols_per_date": len(expected_symbols),
    })

assert train_df["date"].lt(validation_start).all()
assert valid_df["date"].ge(validation_start).all()
assert valid_df["date"].lt(test_start).all()
assert test_df["date"].ge(test_start).all()

assert train_df["label_end_date"].lt(validation_start).all(), (
    "Training outcomes overlap validation"
)
assert valid_df["label_end_date"].lt(test_start).all(), (
    "Validation outcomes overlap test"
)

for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    left_keys = pd.MultiIndex.from_frame(splits[left][["date", "symbol"]])
    right_keys = pd.MultiIndex.from_frame(splits[right][["date", "symbol"]])
    assert left_keys.intersection(right_keys).empty, f"{left}/{right}: shared rows"
    assert set(splits[left]["date"]).isdisjoint(splits[right]["date"]), (
        f"{left}/{right}: shared prediction dates"
    )

display(pd.DataFrame(summary).set_index("split"))

# Show the rows removed because their outcomes crossed a boundary.
excluded_train = df[
    (df["date"] < validation_start) &
    (df["label_end_date"] >= validation_start)
]
excluded_validation = df[
    (df["date"] >= validation_start) &
    (df["date"] < test_start) &
    (df["label_end_date"] >= test_start)
]

print("Excluded training rows:", len(excluded_train))
print("Excluded validation rows:", len(excluded_validation))
display(
    pd.concat([excluded_train, excluded_validation])[
        ["date", "symbol", "label_end_date"]
    ].sort_values(["date", "symbol"]).head(10)
)


,rows,dates,first_date,last_date,latest_label_end,symbols_per_date
split,,,,,,
train,9830,983,2018-01-31,2021-12-23,2021-12-31,10
validation,2460,246,2022-01-03,2022-12-22,2022-12-30,10
test,4970,497,2023-01-03,2024-12-23,2024-12-31,10


Excluded training rows: 50
Excluded validation rows: 50


,date,symbol,label_end_date
983,2021-12-27,AAPL,2022-01-03
2719,2021-12-27,AMGN,2022-01-03
4455,2021-12-27,CAT,2022-01-03
6191,2021-12-27,HD,2022-01-03
7927,2021-12-27,JNJ,2022-01-03
9663,2021-12-27,JPM,2022-01-03
11399,2021-12-27,MSFT,2022-01-03
13135,2021-12-27,NEE,2022-01-03
14871,2021-12-27,WMT,2022-01-03
16607,2021-12-27,XOM,2022-01-03


## 7. Prepare model inputs without fitting preprocessing

Select the explicit input columns as X and target_percentile_5d as y for each split. Retain dates and symbols separately as metadata so predictions can later be evaluated as daily stock rankings. Ensure the feature list is identical and identically ordered across splits.

Raw rolling features may use earlier history across a split boundary: a January validation feature can use December prices that were already available. There is no need to reset rolling windows at each boundary.

A scaler or imputer that learns values across dates must be fitted on training only, then applied unchanged to validation and test. Do not fit it now on the complete table. Same-date percentile transformations have a different information boundary: they can use the eligible stocks known on that date.

In [39]:
X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_valid = valid_df[feature_columns].copy()
y_valid = valid_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()


metadata_columns = ["date", "symbol", "label_end_date"]

train_metadata = train_df[metadata_columns].copy()
valid_metadata = valid_df[metadata_columns].copy()
test_metadata = test_df[metadata_columns].copy()

## 8. Record the baseline and evaluation plan

Plan a simple baseline before fitting an ML model: use past 20-day return as a score, with larger scores ranking above smaller scores. This gives the model a transparent comparison to beat. A reversal baseline can be a later separately documented hypothesis, rather than an extra choice tuned on the test set.

Plan to compare predicted scores with observed target ordering using Spearman correlation across stocks on each date. Report daily values and their summary, not a single pooled correlation across all dates and stocks. Daily five-day outcomes overlap, so naive uncertainty estimates assuming independent rows are inappropriate.

Use validation to compare a limited number of planned models and settings. Keep test outcomes out of that selection. This notebook records the plan; actual baseline scoring and model fitting belong to the next notebook. Portfolio returns, execution timing, and costs will need a later backtest.

In [ ]:
evaluation_plan = {
    "baseline": {
        "score_column": "twenty_day_return",
        "direction": "higher score predicts better future rank",
        "requires_training": False,
    },
    "metric": {
        "name": "daily Spearman rank correlation",
        "group_by": "date",
        "target": target_column,
        "summary": ["mean", "median", "fraction_positive", "undefined_day_count"],
        "constant_scores": "record undefined correlation; do not silently set it to zero",
    },
    "first_model": {
        "name": "StandardScaler followed by Ridge",
        "features": feature_columns,
        "candidate_alphas": [0.1, 1.0, 10.0],
        "preprocessing_fit": "training only",
        "selection": "highest mean daily validation Spearman correlation",
        "tie_break": "larger alpha",
    },
    "test_policy": (
        "Keep the selected training-fitted pipeline fixed for the first test evaluation. "
        "Do not use test outcomes to select settings."
    ),
    "uncertainty": (
        "Five-day labels overlap; do not assume stock-date rows are independent."
    ),
    "scope": "Ranking quality only; execution and trading costs require a later backtest.",
}

display(pd.Series(evaluation_plan["baseline"], name="Baseline"))
display(pd.Series(evaluation_plan["metric"], name="Evaluation metric"))
print("Planned Ridge alpha values:", evaluation_plan["first_model"]["candidate_alphas"])


## 9. Save the split assignments and document the experiment

Save a separate split manifest containing date, symbol, and split membership; document removed boundary rows separately or assign them an excluded status with a reason. Leave the labeled-data export intact. Add a new output path to config.py when implementing this save.

Record boundary dates, target horizon, feature list, target name, strict label-end rules, universe, source file, retained and removed counts, and verification results. A saved manifest lets later modeling reuse precisely the same observations.

Future work can use expanding-window validation to compare several historical periods. Begin with this fixed split so the timing rules and evaluation procedure are clear.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone

from factor_model.config import (
    SPLIT_MANIFEST_PATH,
    SPLIT_SUMMARY_PATH,
    VALIDATION_PLAN_PATH,
)

# Include excluded rows so every source observation has a recorded status.
manifest_parts = []
for name, split in splits.items():
    part = split[["date", "symbol", "label_end_date"]].copy()
    part["split"] = name
    part["exclusion_reason"] = ""
    manifest_parts.append(part)

for excluded, reason in [
    (excluded_train, "training_label_reaches_validation"),
    (excluded_validation, "validation_label_reaches_test"),
]:
    part = excluded[["date", "symbol", "label_end_date"]].copy()
    part["split"] = "excluded"
    part["exclusion_reason"] = reason
    manifest_parts.append(part)

split_manifest = pd.concat(manifest_parts, ignore_index=True)
split_manifest = split_manifest.sort_values(["date", "symbol"])
source_keys = pd.MultiIndex.from_frame(df[["date", "symbol"]])
manifest_keys = pd.MultiIndex.from_frame(split_manifest[["date", "symbol"]])

assert not manifest_keys.has_duplicates, "Manifest contains duplicate keys"
assert len(split_manifest) == len(df), "Manifest row count differs from source"
assert source_keys.difference(manifest_keys).empty, "Source rows missing from manifest"
assert manifest_keys.difference(source_keys).empty, "Manifest has unknown rows"

split_summary = pd.DataFrame(summary)
experiment_record = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_path": str(LABELED_DATA_PATH),
    "source_sha256": hashlib.sha256(LABELED_DATA_PATH.read_bytes()).hexdigest(),
    "validation_start": validation_start.isoformat(),
    "test_start": test_start.isoformat(),
    "target_horizon_observations": 5,
    "label_boundary_rule": "label_end_date strictly before next period start",
    "prediction_time": "after market close",
    "universe": sorted(expected_symbols),
    "retained_rows": {name: len(split) for name, split in splits.items()},
    "excluded_training_rows": len(excluded_train),
    "excluded_validation_rows": len(excluded_validation),
    "checks": "Section 6 assertions passed before export",
    "evaluation_plan": evaluation_plan,
}

SPLIT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
split_manifest.to_csv(SPLIT_MANIFEST_PATH, index=False)
split_summary.to_csv(SPLIT_SUMMARY_PATH, index=False)
with VALIDATION_PLAN_PATH.open("w", encoding="utf-8") as plan_file:
    json.dump(experiment_record, plan_file, indent=2)

print("Saved:", SPLIT_MANIFEST_PATH)
print("Saved:", SPLIT_SUMMARY_PATH)
print("Saved:", VALIDATION_PLAN_PATH)
display(split_manifest["split"].value_counts())


## Validation findings

The initial experiment uses prediction dates in 2018–2021 for training, 2022 for validation, and 2023–2024 for testing. All stocks from a prediction date stay together. Labels ending on or after the following period's boundary are excluded.

Verification on the current labeled dataset retained 9,830 training rows, 2,460 validation rows, and 4,970 test rows. Each retained date contains all ten stocks. Fifty rows were excluded at each of the two boundaries. The split checks confirm unique and disjoint keys, valid chronology, and finite required inputs and targets.

The planned baseline scores stocks by past 20-day return. The first model will use the four raw features with training-fitted StandardScaler and Ridge. Three alpha values (0.1, 1.0, 10.0) will be compared using mean daily validation Spearman correlation. Test results will not select settings; the first test evaluation uses the selected training-fitted pipeline without refitting.

The split manifest includes all source keys, with explicit exclusion reasons. The summary and evaluation plan are saved separately; the plan records a hash of the labeled source file. No model fitting or baseline performance evaluation has been performed in this notebook. Overlapping labels, prior full-history EDA, the small surviving-company universe, and unmodeled trading costs remain limitations.